In [ ]:
import sys
from pathlib import Path

HICOLA_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'scripts' / 'hicola_paths.py').exists())
sys.path.insert(0, str(HICOLA_ROOT / 'scripts'))
import hicola_paths as hp
import analysis as an

import numpy as np
import matplotlib.pyplot as plt

# Hi-COLA vs Abacus_Small

Matter power spectrum and halo mass function of the Hi-COLA runs matched to the AbacusSummit
`small_c000_ph3000` box (500 Mpc/h, c000 cosmology). The Abacus reference has a ΛCDM P(k) only,
so nDGP and the HMFs are shown for the Hi-COLA runs alone, with the nDGP/ΛCDM ratio.

The reference P(k) is made by `jobs/power_spectrum_abacus.sh`. Everything below follows
the parameters in the next cell; Hi-COLA measurements are read from the run folder on scratch,
or from the kept copy in `hicola_data/hicola/` (see `PIPELINE.md`).

In [ ]:
target = 'Abacus_Small'
redshift = 0.5               # the Abacus reference exists at 0.5; the runs also output 1.1 and 0.8
models = {                   # model -> panel title; LCDM first (it is the denominator of nDGP/LCDM)
    'LCDM': r'$\Lambda\mathrm{CDM}$',
    'nDGP_N1.2': r'$\mathrm{nDGP}$',
}
runs = {                     # Hi-COLA runs to show: label -> realizations averaged together
    'COLA (unfixed)': ['s1234'],
    'COLA (paired-fixed)': list(an.PAIRED_FIXED),
}
reference_name = 'Nbody'
reference_boxes = [1]        # Abacus_Small has one box
# Same k range as compare_COLA_NM4096 (the largest k of the COLA_NM4096 reference)
k_max = an.load_table(hp.reference_file('COLA_NM4096', 'LCDM', 1, 1.65, 'PS'))[0].max()
M_min = 1e12                 # lowest halo mass shown
pk_ratio_ylim = (0.8, 2.1)   # y range of the nDGP/LCDM P(k) panel

In [ ]:
# Figure styles: SMALL for multi-panel figures, BIG for one-column figures (shrunk more in a document)
SMALL = dict(figsize=(6.6, 4.35), label=10.5, tick=8.8, legend=8.3, lw=1.45)
BIG = dict(figsize=(6.6, 4.6), label=14.5, tick=12.5, legend=11.5, lw=1.9)
BAND_ALPHA = 0.18
REFERENCE_COLOR = '#1f4e79'
RATIO_REF_COLOR = '0.35'
RUN_COLORS = ['#c47f2b', '#4c8c4a', '#8e5ea2', '#b5446e']
RUN_STYLES = ['--', '-.', ':', (0, (5, 1))]
K_LABEL = r'$k\,[h\,\mathrm{Mpc}^{-1}]$'
M_LABEL = r'$M_{200c}\,[h^{-1}M_\odot]$'


def run_style(i):
    return dict(color=RUN_COLORS[i % len(RUN_COLORS)], ls=RUN_STYLES[i % len(RUN_STYLES)])


def finish(axes, s):
    for ax in np.atleast_1d(axes).ravel():
        if not ax.get_visible():
            continue
        ax.minorticks_on()
        ax.grid(True, which='major', ls=':', color='0.82', alpha=0.75, lw=0.45)
        scale = s['tick'] / SMALL['tick']
        ax.tick_params(axis='both', which='major', direction='in', top=True, right=True, labelsize=s['tick'], length=3.4 * scale, width=0.7 * scale)
        ax.tick_params(axis='both', which='minor', direction='in', top=True, right=True, labelsize=s['tick'] - 0.8, length=2.0 * scale, width=0.55 * scale)
        for spine in ax.spines.values():
            spine.set_linewidth(0.7 * scale)


def legend(ax, s, **kwargs):
    ax.legend(fontsize=s['legend'], frameon=False, handlelength=1.6, borderaxespad=0.3, labelspacing=0.3, **kwargs)

## Load

In [ ]:
lcdm, ndgp = models

# Reference: (kind, model) -> (x, y) where the Abacus box has that measurement
ref = {}
for kind in ('PS', 'HMF'):
    for model in models:
        if all(hp.reference_file(target, model, box, redshift, kind).exists() for box in reference_boxes):
            x, y, _ = an.load_reference(target, model, redshift, kind, boxes=reference_boxes)
            ref[kind, model] = (x, y)
print('Reference measurements:', sorted(ref) or 'none; run jobs/power_spectrum_abacus.sh')

# Hi-COLA measurements: (kind, model, run label) -> (x, y). Runs that do not exist (yet) are skipped.
hic = {}
for kind in ('PS', 'HMF'):
    for label, realizations in runs.items():
        for model in models:
            try:
                hic[kind, model, label] = an.load_hicola(target, model, realizations, redshift, kind)
            except FileNotFoundError as error:
                print(f'Skipping {label} / {model} / {kind}: {error}')

## Matter power spectrum

In [ ]:
# Top: LCDM k^{3/2} P(k) and ratio to the Abacus box. Bottom: nDGP k^{3/2} P(k) and nDGP/LCDM.
s = SMALL
fig, axes = plt.subplots(2, 2, figsize=s['figsize'], dpi=300, constrained_layout=True)
(ax_l, ax_lr), (ax_n, ax_nr) = axes

for ax, model in ((ax_l, lcdm), (ax_n, ndgp)):
    if ('PS', model) in ref:
        k_ref, P_ref = ref['PS', model]
        keep = an.within(k_ref, None, k_max)
        ax.semilogx(k_ref[keep], k_ref[keep]**1.5 * P_ref[keep], lw=s['lw'], color=REFERENCE_COLOR, label=reference_name)
    k_min = ref['PS', lcdm][0].min() if ('PS', lcdm) in ref else None
    for i, label in enumerate(runs):
        if ('PS', model, label) in hic:
            k, P = hic['PS', model, label]
            keep = an.within(k, k_min, k_max)
            ax.semilogx(k[keep], k[keep]**1.5 * P[keep], lw=s['lw'], label=label, **run_style(i))
    ax.set_ylabel(r'$k^{3/2} P(k)$', fontsize=s['label'])
    ax.set_title(models[model], fontsize=s['label'])
    legend(ax, s)

if ('PS', lcdm) in ref:
    k_ref, P_ref = ref['PS', lcdm]
    # The largest-scale Abacus bins have few modes; shot-noise subtraction can push them to <= 0
    keep = an.within(k_ref, None, k_max) & (P_ref > 0)
    for i, label in enumerate(runs):
        if ('PS', lcdm, label) in hic:
            k, P = hic['PS', lcdm, label]
            ax_lr.semilogx(k_ref[keep], an.ratio(k, P, k_ref, P_ref)[keep], lw=s['lw'], label=label, **run_style(i))
    ax_lr.axhline(1.0, color=RATIO_REF_COLOR, lw=0.9, ls=':')
    ax_lr.set_ylabel(rf'$P / P_\mathrm{{{reference_name}}}$', fontsize=s['label'])
    ax_lr.set_title(models[lcdm], fontsize=s['label'])
else:
    ax_lr.set_visible(False)

for i, label in enumerate(runs):
    if ('PS', lcdm, label) in hic and ('PS', ndgp, label) in hic:
        k_l, P_l = hic['PS', lcdm, label]
        k_n, P_n = hic['PS', ndgp, label]
        keep = an.within(k_l, None, k_max)
        ax_nr.semilogx(k_l[keep], an.ratio(k_n, P_n, k_l, P_l)[keep], lw=s['lw'], label=label, **run_style(i))
ax_nr.axhline(1.0, color=RATIO_REF_COLOR, lw=0.9, ls=':')
ax_nr.set_ylabel(r'$P_{\mathrm{nDGP}} / P_{\mathrm{LCDM}}$', fontsize=s['label'])
ax_nr.set_title(r'$\mathrm{nDGP}/\Lambda\mathrm{CDM}$', fontsize=s['label'])
if pk_ratio_ylim:
    ax_nr.set_ylim(*pk_ratio_ylim)

for ax in axes.ravel():
    ax.set_xlabel(K_LABEL, fontsize=s['label'])
finish(axes, s)
plt.show()

## Halo mass function

In [ ]:
# Cumulative HMF per model (with the reference where there is one)
s = SMALL
fig, axes = plt.subplots(1, 2, figsize=(s['figsize'][0], 2.6), dpi=300, constrained_layout=True)
for ax, (model, title) in zip(axes, models.items()):
    if ('HMF', model) in ref:
        M_ref, n_ref = ref['HMF', model]
        keep = (n_ref > 0) & (M_ref >= M_min)
        ax.loglog(M_ref[keep], n_ref[keep], lw=s['lw'], color=REFERENCE_COLOR, label=reference_name)
    for i, label in enumerate(runs):
        if ('HMF', model, label) in hic:
            M, n = hic['HMF', model, label]
            keep = (n > 0) & (M >= M_min)
            ax.loglog(M[keep], n[keep], lw=s['lw'], label=label, **run_style(i))
    ax.set_xlabel(M_LABEL, fontsize=s['label'])
    ax.set_ylabel(r'$n(>M)\,[h^3\,\mathrm{Mpc}^{-3}]$', fontsize=s['label'])
    ax.set_title(title, fontsize=s['label'])
    legend(ax, s)
finish(axes, s)
plt.show()

In [ ]:
# nDGP / LCDM HMF, one-column figure
s = BIG
fig, ax = plt.subplots(figsize=s['figsize'], dpi=300, constrained_layout=True)
for i, label in enumerate(runs):
    if ('HMF', lcdm, label) in hic and ('HMF', ndgp, label) in hic:
        M_l, n_l = hic['HMF', lcdm, label]
        M_n, n_n = hic['HMF', ndgp, label]
        r = an.ratio(M_n, n_n, M_l, n_l)
        keep = np.isfinite(r) & (M_l >= M_min)
        ax.semilogx(M_l[keep], r[keep], lw=s['lw'], label=label, **run_style(i))
ax.axhline(1.0, color='0.4', lw=1.1, ls='--')
ax.set_xlabel(M_LABEL, fontsize=s['label'])
ax.set_ylabel(r'$n_{\mathrm{nDGP}}(>M) / n_{\mathrm{LCDM}}(>M)$', fontsize=s['label'])
legend(ax, s)
finish(ax, s)
plt.show()